In [1]:

import numpy as np
import pandas as pd

In [4]:

data_04_20 = pd.read_excel('101-20200501/042020B1.xlsx')

data_05_20 = pd.read_excel('101-20200601/052020B1.xlsx')

data_04_21 = pd.read_excel('101-20210501/042021B1.xlsx')

data_05_21 = pd.read_excel('101-20210601/052021B1.xlsx')

banks_names = pd.read_excel('101-20210601/052021N1.xlsx')

In [161]:

data = [data_04_20, data_05_20, data_04_21, data_05_21, banks_names]

pos_values = ['70601', '70602', '70603', '70604', '70605', '70613', '70615']

neg_values = ['70606', '70607', '70608', '70609', '70610', '70611', '70614', '70616']

In [162]:

def columns(data_list):
    
    for i in range(len(data_list) - 1):
        
        data_list[i] = data_list[i][['REGN','NUM_SC','IITG']]

    return data_list



def filtering(data_list):
    
    for i in range(len(data_list) - 1):
        
        data_list[i] = data_list[i][data_list[i]['NUM_SC'].isin(pos_values + neg_values)]
        
    return data_list



data = filtering(columns(data))

In [163]:

merged_data_20 = pd.merge(data[0], data[1], on=['REGN','NUM_SC'], how='outer').fillna(0)

merged_data_21 = pd.merge(data[2], data[3], on=['REGN','NUM_SC'], how='outer').fillna(0)

In [164]:

merged_data_20['DIFF'] = merged_data_20.iloc[:,3] - merged_data_20.iloc[:,2]

merged_data_21['DIFF'] = merged_data_21.iloc[:,3] - merged_data_21.iloc[:,2]

In [165]:

def profit(df):
    
    p = 0
    
    for i in range(len(df)):
        
        if df.iloc[i,1] in pos_values:
            
            p = p + df.iloc[i,4]
            
        elif df.iloc[i,1] in neg_values:
                
            p = p - df.iloc[i,4]
                
    return p


banks_profit_20 = merged_data_20.groupby(merged_data_20['REGN']).apply(profit)

banks_profit_21 = merged_data_21.groupby(merged_data_21['REGN']).apply(profit)

In [166]:

banks_profit_with_names_21 = pd.merge(banks_profit_21.reset_index(), data[4], on='REGN').drop(['REGN','PRIZ', 'PRIZ_P'], axis=1)

banks_profit_with_names_21.rename(columns={0: 'PROFIT'}, inplace=True)

banks_profit_with_names_21 = banks_profit_with_names_21.sort_values(by='PROFIT', ascending=False)


top_ten_profit = banks_profit_with_names_21.reindex(columns=['NAME_B','PROFIT'])[:10]

In [167]:

banks_profit_20.name = 'PROFIT_20'

banks_profit_21.name = 'PROFIT_21'


profits = pd.merge(banks_profit_20, banks_profit_21, left_index=True, right_index=True, how='inner')

profits['RATE'] = (profits['PROFIT_21'] - profits['PROFIT_20']) / profits['PROFIT_20'] * 100

profits = pd.merge(profits.reset_index(), data[4], on='REGN').drop(['REGN','PROFIT_20','PROFIT_21','PRIZ', 'PRIZ_P'], axis=1)

profits = profits.sort_values(by='RATE', ascending=False)


top_ten_rate = profits.reindex(columns=['NAME_B','RATE'])[:10]

In [169]:

top_ten_profit.index = range(len(top_ten_profit.index))

top_ten_rate.index = range(len(top_ten_rate.index))

results = pd.concat([top_ten_profit, top_ten_rate], axis=1, keys=['top_ten_profit','top_ten_rate'])

In [171]:

results.to_excel("results.xlsx")